# traffic-cv: обучение на GPU

Секреты Colab (значок ключа слева): `DAGSHUB_USER`, `DAGSHUB_TOKEN`.
Датасет (`data/processed/detrac`) заранее загружен в DVC-хранилище командой `dvc push` с локальной машины.

In [ ]:
!nvidia-smi
!git clone https://github.com/jestersw/trafic-cv.git
%cd trafic-cv
!pip install -q -e ".[train,track,eda,data]"

In [ ]:
import os

from google.colab import userdata

user = userdata.get("DAGSHUB_USER")
token = userdata.get("DAGSHUB_TOKEN")
repo = f"https://dagshub.com/{user}/trafic-cv"

os.environ["MLFLOW_TRACKING_URI"] = f"{repo}.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = user
os.environ["MLFLOW_TRACKING_PASSWORD"] = token

!dvc remote add -f -d dagshub {repo}.dvc
!dvc remote modify dagshub --local auth basic
!dvc remote modify dagshub --local user {user}
!dvc remote modify dagshub --local password {token}

## Данные

In [ ]:
!dvc pull data/processed/detrac
!ls data/processed/detrac

## Детекция (этап 2) и трекинг (этап 3)

In [ ]:
!dvc repro -s train_baseline train_finetune compare_detection eval_tracking

In [ ]:
!cat reports/detection/compare.md
!cat reports/tracking/compare.md

## Сохранить результаты

Модели уходят в DVC-хранилище. `dvc.lock` и отчёты нужно скачать и положить в локальный репозиторий, чтобы закоммитить.

In [ ]:
!dvc push
!zip -qr results.zip dvc.lock reports
from google.colab import files

files.download("results.zip")